In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

## DATA LOADING ##


In [ ]:
df = pd.read_csv('AgeDataset-V1.csv')
df

In [ ]:
df.columns=df.columns.str.lower().str.strip().str.replace(' ' , '_')

## DATA UNDERSTANDING

In [ ]:
df.head()

In [ ]:
df.drop(columns=['id','short_description' , 'birth_year' , 'death_year'] , inplace=True) ## age of death is the subtraction of the death year from the birth_year
df.info()

## DATA CLEANING

In [ ]:
(df.isna().mean()*100).sort_values(ascending=False)

In [ ]:
df['manner_of_death'].fillna(df['manner_of_death'].mode()[0],inplace=True) ## filling the missing values

In [ ]:
df['country'].fillna(df['country'].mode()[0],inplace=True)   ## filling the missing values

In [ ]:
df['occupation'].fillna(df['occupation'].mode()[0],inplace=True)  ## filling the missing values

In [ ]:
df['gender'].fillna(df['gender'].mode()[0],inplace=True)   ## filling the missing values

In [ ]:
def outlier_percentage(series):
    series=series.dropna()
    
    q1=series.quantile(0.25)
    q3=series.quantile(0.75)
    iqr=q3-q1
    lower=q1-1.5*iqr
    upper=q3+1.5*iqr
    outliers=series[(series<lower)|(series>upper)]
    
    return round(len(outliers)/len(series)*100,2)

fill=['age_of_death']
outlier_report={col:outlier_percentage(df[col])
                for col in fill}
outlier_report

In [ ]:
df['age_of_death'].fillna(df['age_of_death'].mean(),inplace = True)  ## filling the missing values

In [ ]:
(df.isna().mean()*100).sort_values(ascending=False)

In [ ]:
df.duplicated().sum()


In [ ]:
df.drop_duplicates(inplace=True) ## removing duplicates

In [ ]:
df.duplicated().sum()

In [ ]:
clean_df = df
clean_df.to_csv('cleaned_df.csv' , index = False)

## DATA VISUALIZATION

In [ ]:
df.head()

In [ ]:
fig1 =px.histogram(df, x='age_of_death' , title='Distribution of Age at Death')
fig1

In [ ]:
fig2=px.box(df, y='age_of_death')
fig2

In [ ]:
fig3 =px.box(df, x='gender', y='age_of_death' , title= 'Age at Death by Gender')
fig3

In [ ]:
df.head(1)

In [ ]:

max_age_by_country = df.groupby('country')['age_of_death'].max().sort_values(ascending=False).head(10)
max_age_df = max_age_by_country.reset_index() 
max_age_df.columns = ['country', 'max_age']

px.bar(max_age_df, x='country', y='max_age',
       title='Top 10 Countries by Maximum Age at Death')

In [ ]:
Average_Age_by_country = df.groupby('country')['age_of_death'].mean().sort_values(ascending=False).head(10)
Average_Age_df = Average_Age_by_country.reset_index() 
Average_Age_df = Average_Age_by_country.reset_index() 
Average_Age_df.columns = ['country', 'max_age']

px.bar(max_age_df, x='country', y='max_age',
       title='Top 10 Countries by Average Age at Death')


In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import plotly.express as px

df1 = pd.read_csv('cleaned_df.csv')

st.title("Age Dataset Visualization")

min_age = int(df1['age_of_death'].min())
max_age = int(df1['age_of_death'].max())

age_range = st.slider(
    "Select Age Range:",
    min_value=min_age,
    max_value=max_age,
    value=(min_age, max_age)
)


filtered_df = df1[(df1['age_of_death'] >= age_range[0]) & (df1['age_of_death'] <= age_range[1])]


if st.checkbox("Show Age Distribution Histogram", value=True):
    st.subheader("Distribution of Age at Death")
    fig1 = px.histogram(filtered_df, x='age_of_death', nbins=30,
                        title=f'Distribution of Age at Death ({age_range[0]} - {age_range[1]})')
    st.plotly_chart(fig1)


option = st.selectbox(
    "Show Top 10 or Bottom 10 Countries by Max Age?",
    ("Top 10", "Bottom 10")
)

if option == "Top 10":
    max_age_by_country = df1.groupby('country')['age_of_death'].max().sort_values(ascending=False).head(10)
else:
    max_age_by_country = df1.groupby('country')['age_of_death'].max().sort_values(ascending=True).head(10)

max_age_df = max_age_by_country.reset_index()
max_age_df.columns = ['country', 'max_age']

st.subheader(f"{option} Countries by Maximum Age at Death")
fig2 = px.bar(max_age_df, x='country', y='max_age',
              title=f"{option} Countries by Maximum Age at Death")
st.plotly_chart(fig2)

In [ ]:
! streamlit run app.py 